<a href="https://colab.research.google.com/github/alexwmackay/numerai_models/blob/main/numerai_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0.0 Import Packages**

In [1]:
!pip install pandas scikit-learn numerapi

In [2]:
import pandas as pd
import numpy as np
import numerapi
import gc
import json
import os
from datetime import datetime
from pathlib import Path
import sklearn.linear_model
import matplotlib.pyplot as plt
from numerapi import NumerAPI
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def load_numerai_data(
    version=None,
    cache_dir="/content/drive/My Drive/numerai_data",
    feature_set="all",
    force_refresh=False,
    colab=True,
    public_id=None,
    secret_key=None,
):

    # Setup
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    train_path = f"{cache_dir}/train.parquet"
    val_path = f"{cache_dir}/validation.parquet"
    features_path = f"{cache_dir}/features.json"
    meta_path = f"{cache_dir}/metadata.json"

    # Get credentials
    if public_id is None or secret_key is None:
        if colab:
            from google.colab import userdata
            public_id = userdata.get('NUMERAI_PUBLIC_ID')
            secret_key = userdata.get('NUMERAI_SECRET_KEY')
        else:
            raise ValueError("Must provide public_id and secret_key for local execution")

    napi = NumerAPI(public_id=public_id, secret_key=secret_key)

    if version is None:
        try:
            available_datasets = napi.get_available_datasets()
            version = available_datasets[0] if available_datasets else "v5.3"
            print(f"🔍 Auto-detected version: {version}")
        except Exception as e:
            print(f"⚠️  Could not auto-detect version, falling back to v5.3: {e}")
            version = "v5.3"

    # Check if update needed
    needs_update = force_refresh or not all([
        os.path.exists(train_path),
        os.path.exists(val_path),
        os.path.exists(features_path)
    ])

    if not needs_update and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)

        needs_update = False

    # Download if needed
    if needs_update:
        print(f"📥 Downloading Numerai {version} data...")
        napi.download_dataset(f"{version}/train.parquet", dest_path=train_path)
        napi.download_dataset(f"{version}/validation.parquet", dest_path=val_path)
        napi.download_dataset(f"{version}/features.json", dest_path=features_path)

        # Save metadata
        metadata = {
            'version': version,
            'feature_set': feature_set,
            'updated': datetime.now().isoformat(),
            'cached': True
        }
        with open(meta_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"✓ Data cached to: {cache_dir}")
    else:
        with open(meta_path) as f:
            metadata = json.load(f)
        print(f"✓ Using cached data (updated: {metadata.get('updated', 'unknown')})")

    # Load features
    with open(features_path) as f:
        features = json.load(f)

    feature_cols = features['feature_sets'][feature_set]
    cols = feature_cols + ['target', 'era', 'id']

    # Load datasets
    print(f"Loading train data...")
    train_data = pd.read_parquet(train_path, columns=cols)
    train_data[feature_cols] = train_data[feature_cols].astype('float32')
    print(f"  ✓ Train: {train_data.shape} | {train_data.memory_usage(deep=True).sum() / 1e6:.1f}MB")

    print(f"Loading validation data...")
    val_data = pd.read_parquet(val_path, columns=cols)
    val_data[feature_cols] = val_data[feature_cols].astype('float32')
    print(f"  ✓ Val: {val_data.shape} | {val_data.memory_usage(deep=True).sum() / 1e6:.1f}MB")

    # Cleanup
    gc.collect()

    # Return structured output
    return {
        'train': train_data,
        'val': val_data,
        'features': feature_cols,
        'metadata': metadata
    }

**1.0 Data Download**

In [ ]:
data = load_numerai_data()

train = data['train']

print(f"Loaded version: {data['metadata']['version']}")

⚠️  Could not auto-detect version, falling back to v5.3: 'NumerAPI' object has no attribute 'get_available_datasets'
✓ Using cached data (updated: 2026-08-23T11:22:37.031951)
Loading train data...


**2.0 Visualise**
